In [1]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pandas as pd

# Download required NLTK data packages
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')

print("NLTK Resources Downloaded Successfully!")

NLTK Resources Downloaded Successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\omp72\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\omp72\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\omp72\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\omp72\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# Dictionary for expanding English contractions
CONTRACTIONS = {
    "can't": "cannot", "won't": "will not", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would",
    "'ll": " will", "'t": " not", "'ve": " have", "'m": " am"
}

def expand_contractions(text):
    """Expands words like don't -> do not, i'm -> i am"""
    for contraction, expansion in CONTRACTIONS.items():
        text = text.replace(contraction, expansion)
    return text

# Initialize Lemmatizer and Stopwords list
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

print("Helper functions ready!")

Helper functions ready!


In [3]:
def preprocess_text(text, remove_stop_words=True, use_lemmatization=True):
    """
    Complete NLP Cleaning Pipeline:
    1. Lowercase
    2. Remove HTML & URLs
    3. Expand Contractions
    4. Remove Punctuation & Special Characters
    5. Tokenize & Remove Stopwords (optional)
    6. Lemmatize words (optional)
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove URLs & HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    
    # 3. Expand contractions
    text = expand_contractions(text)
    
    # 4. Remove punctuation & special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # 5. Tokenization (split by whitespace)
    words = text.split()
    
    # 6. Stopword removal (Optional)
    if remove_stop_words:
        words = [w for w in words if w not in stop_words]
        
    # 7. Lemmatization (Optional)
    if use_lemmatization:
        words = [lemmatizer.lemmatize(w) for w in words]
        
    return " ".join(words)

In [4]:
# Test on sample noisy questions
sample_q1 = "How can I learn Python quickly??? Visit https://python.org! I can't wait!"
sample_q2 = "What's the best method to become good at Python & Data Science?"

print("--- BEFORE ---")
print("Q1 Raw:", sample_q1)
print("Q2 Raw:", sample_q2)

print("\n--- AFTER PREPROCESSING ---")
print("Q1 Cleaned:", preprocess_text(sample_q1))
print("Q2 Cleaned:", preprocess_text(sample_q2))

--- BEFORE ---
Q1 Raw: How can I learn Python quickly??? Visit https://python.org! I can't wait!
Q2 Raw: What's the best method to become good at Python & Data Science?

--- AFTER PREPROCESSING ---
Q1 Cleaned: learn python quickly visit cannot wait
Q2 Cleaned: best method become good python data science


In [5]:
# Load dataset
df = pd.read_csv('../data/quora_ques.csv').dropna(subset=['question1', 'question2'])

# Take a sample of 1000 rows for quick testing
sample_df = df.head(10).copy()

# Apply cleaning to both question columns
sample_df['q1_clean'] = sample_df['question1'].apply(preprocess_text)
sample_df['q2_clean'] = sample_df['question2'].apply(preprocess_text)

# Display Raw vs Cleaned side-by-side
sample_df[['question1', 'q1_clean', 'question2', 'q2_clean']].head(5)

,question1,q1_clean,question2,q2_clean
0,What is the step by step guide to invest in sh...,step step guide invest share market india,What is the step by step guide to invest in sh...,step step guide invest share market
1,What is the story of Kohinoor (Koh-i-Noor) Dia...,story kohinoor kohinoor diamond,What would happen if the Indian government sto...,would happen indian government stole kohinoor ...
2,How can I increase the speed of my internet co...,increase speed internet connection using vpn,How can Internet speed be increased by hacking...,internet speed increased hacking dns
3,Why am I mentally very lonely? How can I solve...,mentally lonely solve,Find the remainder when [math]23^{24}[/math] i...,find remainder math2324math divided 2423
4,"Which one dissolve in water quikly sugar, salt...",one dissolve water quikly sugar salt methane c...,Which fish would survive in salt water?,fish would survive salt water
